# Random Forest Algorithm

In this notebook, we'll explore the Random Forest algorithm, which is an ensemble method based on bagging and feature randomness. We'll cover the concepts of bagging, feature randomness, and Out-of-Bag (OOB) error estimation.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, load_wine, make_regression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Understanding Random Forest: A Comprehensive Overview

Random Forest is an ensemble learning method that operates by constructing multiple decision trees during training and outputting the class that is the mode of the classes (classification) or mean prediction (regression) of the individual trees.

### Key Concepts:
1. **Bagging (Bootstrap Aggregating)**: Training multiple models on different subsets of data
2. **Feature Randomness**: Using a random subset of features at each split
3. **Out-of-Bag (OOB) Error**: Using unused samples for internal validation

## 1. Bagging: Bootstrap Aggregating

Bagging works by creating multiple bootstrap samples from the original dataset, training a model on each sample, and then combining the predictions. Let's implement this concept step-by-step.

In [ ]:
# Generate a synthetic dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=8, 
                          n_redundant=2, n_clusters_per_class=1, random_state=42)

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")

In [ ]:
# Demonstrate bagging concept manually
def demonstrate_bagging(X_train, y_train, X_test, y_test, n_estimators=10):
    """Demonstrate the concept of bagging"""
    
    # Single decision tree
    single_tree = DecisionTreeClassifier(random_state=42)
    single_tree.fit(X_train, y_train)
    single_tree_score = single_tree.score(X_test, y_test)
    
    # Manual bagging implementation
    n_samples = X_train.shape[0]
    predictions = np.zeros((n_estimators, X_test.shape[0]))
    
    for i in range(n_estimators):
        # Create bootstrap sample
        bootstrap_indices = np.random.choice(n_samples, size=n_samples, replace=True)
        X_bootstrap = X_train[bootstrap_indices]
        y_bootstrap = y_train[bootstrap_indices]
        
        # Train a decision tree on bootstrap sample
        tree = DecisionTreeClassifier(random_state=42+i)
        tree.fit(X_bootstrap, y_bootstrap)
        
        # Make predictions
        predictions[i] = tree.predict(X_test)
    
    # Aggregate predictions (majority vote)
    bagged_predictions = []
    for j in range(X_test.shape[0]):
        votes = predictions[:, j]
        # Count votes and take majority
        unique, counts = np.unique(votes, return_counts=True)
        majority_vote = unique[np.argmax(counts)]
        bagged_predictions.append(majority_vote)
    
    bagged_predictions = np.array(bagged_predictions)
    bagged_score = accuracy_score(y_test, bagged_predictions)
    
    print(f"Single Decision Tree Accuracy: {single_tree_score:.4f}")
    print(f"Manual Bagging Accuracy: {bagged_score:.4f}")
    
    return single_tree_score, bagged_score

# Demonstrate bagging
single_acc, bagged_acc = demonstrate_bagging(X_train, y_train, X_test, y_test)

In [ ]:
# Compare with scikit-learn's BaggingClassifier
from sklearn.ensemble import BaggingClassifier

bagging_clf = BaggingClassifier(
    base_estimator=DecisionTreeClassifier(),
    n_estimators=10,
    random_state=42
)

bagging_clf.fit(X_train, y_train)
bagging_score = bagging_clf.score(X_test, y_test)

print(f"Scikit-learn Bagging Classifier Accuracy: {bagging_score:.4f}")

## 2. Feature Randomness in Random Forest

Unlike simple bagging, Random Forest adds an extra layer of randomness by considering only a random subset of features at each split. This reduces correlation between trees and improves generalization.

In [ ]:
# Demonstrate the impact of feature randomness
def compare_feature_subsets(X_train, y_train, X_test, y_test):
    feature_subset_options = ['auto', 'sqrt', 'log2', 0.5, 0.8, None]
    results = []
    
    for max_features in feature_subset_options:
        rf = RandomForestClassifier(
            n_estimators=100,
            max_features=max_features,
            random_state=42,
            n_jobs=-1
        )
        
        rf.fit(X_train, y_train)
        train_score = rf.score(X_train, y_train)
        test_score = rf.score(X_test, y_test)
        
        # Calculate number of features used
        if max_features == 'auto' or max_features == 'sqrt':
            n_features_used = int(np.sqrt(X_train.shape[1]))
        elif max_features == 'log2':
            n_features_used = int(np.log2(X_train.shape[1]))
        elif max_features is None:
            n_features_used = X_train.shape[1]
        elif isinstance(max_features, float):
            n_features_used = int(max_features * X_train.shape[1])
        else:
            n_features_used = max_features
        
        results.append({
            'max_features': str(max_features),
            'n_features_used': n_features_used,
            'train_accuracy': train_score,
            'test_accuracy': test_score,
            'overfitting_gap': train_score - test_score
        })
    
    return pd.DataFrame(results)

# Compare different feature subset strategies
feature_comparison = compare_feature_subsets(X_train, y_train, X_test, y_test)
print("Feature Subset Strategy Comparison:")
print(feature_comparison.round(4))

## 3. Out-of-Bag (OOB) Error Estimation

OOB error is an internal validation technique used in Random Forest. Since each tree is trained on a bootstrap sample, about 1/3 of the original samples are left out (out-of-bag). These can be used to estimate the model's performance without needing a separate validation set.

In [ ]:
# Demonstrate OOB error calculation
def calculate_oob_error_manually(X, y, n_estimators=100):
    """Manually calculate OOB error to understand the concept"""
    n_samples = X.shape[0]
    oob_predictions = np.zeros((n_samples, 2))  # For binary classification
    oob_counts = np.zeros(n_samples)
    
    for i in range(n_estimators):
        # Create bootstrap sample
        bootstrap_indices = np.random.choice(n_samples, size=n_samples, replace=True)
        X_bootstrap = X[bootstrap_indices]
        y_bootstrap = y[bootstrap_indices]
        
        # Find out-of-bag samples
        unique, counts = np.unique(bootstrap_indices, return_counts=True)
        oob_indices = np.setdiff1d(np.arange(n_samples), unique)
        
        # Train a tree
        tree = DecisionTreeClassifier(random_state=42+i)
        tree.fit(X_bootstrap, y_bootstrap)
        
        # Predict on OOB samples
        if len(oob_indices) > 0:
            oob_pred = tree.predict(X[oob_indices])
            for j, idx in enumerate(oob_indices):
                oob_predictions[idx, int(oob_pred[j])] += 1
                oob_counts[idx] += 1
    
    # Calculate OOB error
    valid_oob = oob_counts > 0
    final_predictions = np.argmax(oob_predictions[valid_oob], axis=1)
    true_labels = y[valid_oob]
    
    oob_error = 1 - accuracy_score(true_labels, final_predictions)
    coverage = np.sum(valid_oob) / n_samples
    
    return oob_error, coverage

# Calculate manual OOB error
manual_oob_error, coverage = calculate_oob_error_manually(X_train, y_train)
print(f"Manual OOB Error: {manual_oob_error:.4f}")
print(f"OOB Coverage: {coverage:.4f}")

# Compare with sklearn's built-in OOB
rf_with_oob = RandomForestClassifier(
    n_estimators=100,
    oob_score=True,
    random_state=42
)

rf_with_oob.fit(X_train, y_train)
sklearn_oob_error = 1 - rf_with_oob.oob_score_
print(f"Sklearn OOB Error: {sklearn_oob_error:.4f}")
print(f"Test Set Accuracy: {rf_with_oob.score(X_test, y_test):.4f}")

## 4. Hyperparameter Tuning for Random Forest

Let's explore important hyperparameters and their impact on model performance.

In [ ]:
# Analyze the impact of different hyperparameters
def analyze_hyperparameters():
    # Vary number of estimators
    n_estimators_range = [10, 25, 50, 100, 200]
    estimator_results = []
    
    for n_est in n_estimators_range:
        rf = RandomForestClassifier(
            n_estimators=n_est,
            random_state=42,
            oob_score=True
        )
        rf.fit(X_train, y_train)
        
        estimator_results.append({
            'n_estimators': n_est,
            'train_accuracy': rf.score(X_train, y_train),
            'test_accuracy': rf.score(X_test, y_test),
            'oob_accuracy': rf.oob_score_,
            'training_time': 0  # Placeholder
        })
    
    est_results_df = pd.DataFrame(estimator_results)
    
    # Plot results
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(est_results_df['n_estimators'], est_results_df['train_accuracy'], 
             label='Training Accuracy', marker='o')
    plt.plot(est_results_df['n_estimators'], est_results_df['test_accuracy'], 
             label='Test Accuracy', marker='s')
    plt.plot(est_results_df['n_estimators'], est_results_df['oob_accuracy'], 
             label='OOB Accuracy', marker='^')
    plt.xlabel('Number of Estimators')
    plt.ylabel('Accuracy')
    plt.title('Accuracy vs Number of Estimators')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.plot(est_results_df['n_estimators'], 
             est_results_df['train_accuracy'] - est_results_df['test_accuracy'], 
             label='Overfitting Gap', marker='D')
    plt.xlabel('Number of Estimators')
    plt.ylabel('Accuracy Difference')
    plt.title('Overfitting Gap vs Number of Estimators')
    plt.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return est_results_df

# Analyze number of estimators
estimator_analysis = analyze_hyperparameters()
print("Estimator Analysis Results:")
print(estimator_analysis[['n_estimators', 'test_accuracy', 'oob_accuracy']].round(4))

In [ ]:
# Grid search for optimal hyperparameters
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2', None]
}

rf_grid = RandomForestClassifier(random_state=42, n_jobs=-1)
grid_search = GridSearchCV(
    rf_grid, 
    param_grid, 
    cv=5, 
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("Performing grid search...")
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

# Evaluate best model
best_rf = grid_search.best_estimator_
train_acc = best_rf.score(X_train, y_train)
test_acc = best_rf.score(X_test, y_test)
oob_acc = best_rf.oob_score_

print(f"Best model - Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}, OOB Acc: {oob_acc:.4f}")

## 5. Feature Importance Analysis

Random Forest provides a way to measure feature importance, which can help us understand which features are most predictive.

In [ ]:
# Feature importance analysis
rf_for_importance = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_features='sqrt'
)
rf_for_importance.fit(X_train, y_train)

# Get feature importances
feature_importance = rf_for_importance.feature_importances_
feature_names = [f'Feature_{i}' for i in range(len(feature_importance))]

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

# Plot feature importances
plt.figure(figsize=(12, 6))
sns.barplot(data=importance_df.head(10), x='Importance', y='Feature')
plt.title('Top 10 Most Important Features')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("Top 10 Most Important Features:")
print(importance_df.head(10))

## 6. Real-World Example: Wine Dataset

Let's apply Random Forest to a real-world multiclass classification problem.

In [ ]:
# Load the wine dataset
wine_data = load_wine()
X_wine, y_wine = wine_data.data, wine_data.target

# Split the data
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42
)

print(f"Wine dataset shape: {X_wine.shape}")
print(f"Number of classes: {len(np.unique(y_wine))}")
print(f"Classes: {wine_data.target_names}")
print(f"Features: {list(wine_data.feature_names[:5])}...")  # Show first 5 features

In [ ]:
# Apply Random Forest to wine dataset
def evaluate_random_forest_on_dataset(X_train, X_test, y_train, y_test, dataset_name):
    print(f"\n--- Results for {dataset_name} ---")
    
    # Baseline: single decision tree
    dt = DecisionTreeClassifier(random_state=42)
    dt.fit(X_train, y_train)
    dt_score = dt.score(X_test, y_test)
    
    # Random Forest with default parameters
    rf_default = RandomForestClassifier(random_state=42, n_estimators=100)
    rf_default.fit(X_train, y_train)
    rf_default_score = rf_default.score(X_test, y_test)
    
    # Random Forest with optimized parameters
    rf_optimized = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        max_features='sqrt',
        random_state=42,
        oob_score=True
    )
    rf_optimized.fit(X_train, y_train)
    rf_optimized_score = rf_optimized.score(X_test, y_test)
    
    # Compare performances
    print(f"Single Decision Tree Accuracy: {dt_score:.4f}")
    print(f"Default Random Forest Accuracy: {rf_default_score:.4f}")
    print(f"Optimized Random Forest Accuracy: {rf_optimized_score:.4f}")
    print(f"OOB Accuracy (Optimized): {rf_optimized.oob_score_:.4f}")
    
    # Detailed classification report for best model
    y_pred = rf_optimized.predict(X_test)
    print(f"\nClassification Report for Optimized Random Forest:")
    print(classification_report(y_test, y_pred, target_names=wine_data.target_names))
    
    return rf_optimized

# Evaluate Random Forest on wine dataset
best_wine_rf = evaluate_random_forest_on_dataset(
    X_train_w, X_test_w, y_train_w, y_test_w, "Wine Dataset"
)

## 7. Random Forest for Regression

Random Forest can also be used for regression tasks. Let's explore this application.

In [ ]:
# Generate regression dataset
X_reg, y_reg = make_regression(n_samples=1000, n_features=10, n_informative=8, 
                               noise=0.1, random_state=42)

# Split the data
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Compare different approaches
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression

# Models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

# Train and evaluate models
regression_results = []
for name, model in models.items():
    model.fit(X_train_r, y_train_r)
    
    train_pred = model.predict(X_train_r)
    test_pred = model.predict(X_test_r)
    
    train_mse = mean_squared_error(y_train_r, train_pred)
    test_mse = mean_squared_error(y_test_r, test_pred)
    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)
    
    regression_results.append({
        'Model': name,
        'Train_RMSE': train_rmse,
        'Test_RMSE': test_rmse,
        'Overfitting_Gap': train_rmse - test_rmse
    })

regression_df = pd.DataFrame(regression_results)
print("Regression Model Comparison:")
print(regression_df.round(4))

## 8. Advanced Random Forest Concepts

### Understanding Random Forest Parameters:

- `n_estimators`: Number of trees in the forest
- `max_features`: Number of features to consider at each split
- `max_depth`: Maximum depth of the tree
- `min_samples_split`: Minimum samples required to split a node
- `min_samples_leaf`: Minimum samples required at a leaf node
- `bootstrap`: Whether bootstrap samples are used when building trees
- `oob_score`: Whether to use out-of-bag samples to estimate generalization accuracy

In [ ]:
# Effect of bootstrap sampling
def compare_bootstrap_sampling(X_train, y_train, X_test, y_test):
    results = []
    
    for bootstrap in [True, False]:
        rf = RandomForestClassifier(
            n_estimators=100,
            max_features='sqrt',
            random_state=42,
            bootstrap=bootstrap,
            oob_score=bootstrap  # OOB only makes sense with bootstrap=True
        )
        rf.fit(X_train, y_train)
        
        train_acc = rf.score(X_train, y_train)
        test_acc = rf.score(X_test, y_test)
        oob_acc = rf.oob_score_ if bootstrap else np.nan
        
        results.append({
            'Bootstrap': bootstrap,
            'Train_Accuracy': train_acc,
            'Test_Accuracy': test_acc,
            'OOB_Accuracy': oob_acc
        })
    
    return pd.DataFrame(results)

bootstrap_comparison = compare_bootstrap_sampling(X_train, y_train, X_test, y_test)
print("Effect of Bootstrap Sampling:")
print(bootstrap_comparison.round(4))

## Key Takeaways

1. **Bagging**: Random Forest uses bootstrap aggregating to reduce variance and improve stability
2. **Feature Randomness**: At each split, only a random subset of features is considered, reducing correlation between trees
3. **Out-of-Bag Error**: Provides an internal validation mechanism without requiring a separate validation set
4. **Robust Performance**: Random Forest generally performs well with default parameters and is robust to outliers
5. **Feature Importance**: Provides insights into which features are most predictive
6. **Works for Both Classification and Regression**: Can handle both types of problems effectively
7. **Less Prone to Overfitting**: Compared to individual decision trees, Random Forest is less likely to overfit

Random Forest is a powerful ensemble method that combines the benefits of multiple decision trees while mitigating their individual weaknesses. It's widely used in practice due to its robustness and good performance across various types of datasets.